In [ ]:
import run
import chephy_model as cpm
import numpy as np
import torch
from torch_geometric.data import HeteroData
from torch_geometric.transforms import ToUndirected
import pandas as pd
from sklearn.model_selection import train_test_split


In [ ]:
def get_epitope_features(epitopes, atchley_dict):
    # Create epitope features (mean of Atchley vectors per sequence)
    epitope_features = []
    for epitope in epitopes:
        vecs = [list(atchley_dict[aa].values()) for aa in epitope if aa in atchley_dict]
        if vecs:
            mean_vec = np.mean(vecs, axis=0)
        else:
            mean_vec = np.zeros(len(next(iter(atchley_dict.values()))))
        epitope_features.append(mean_vec)

    epitope_features = torch.tensor(epitope_features, dtype=torch.float)
    return epitope_features

In [ ]:
def get_tcr_features(sequences_list, atchley_dict):
    # Use Atchley vectors as node features for TCRs
    tcr_features = []
    for seq in sequences_list:
        vecs = [list(atchley_dict[aa].values()) for aa in seq if aa in atchley_dict]
        tcr_features.append(np.concatenate(vecs))
    tcr_features = torch.tensor(tcr_features, dtype=torch.float)  # [num_tcr, 8*5]
    return tcr_features

In [ ]:
def build_hetero_graph(data, chephy_matrix, hamming_matrix, sequences_list, epitopes):
    # Load Atchley dictionary
    atchley_df = pd.read_csv("/home/dsi/chenbis/repos/sol_lab/files/atchley.csv")
    atchley_dict = atchley_df.set_index('amino.acid').to_dict(orient='index')    

    epitope_features = get_epitope_features(epitopes, atchley_dict)
    tcr_features = get_tcr_features(sequences_list, atchley_dict)

    # Get edges from the data
    edge_index = torch.tensor(data[['tcr_index', 'epitope_index']].dropna().astype(int).values.T, dtype=torch.long)  # shape [2, num_edges]
    
    # Convert chephy_matrix (distance) into edge index for TCR similarity
    threshold = 1  # You can tune this value

    rows, cols = np.where((chephy_matrix < threshold) & (chephy_matrix > 0))  # Exclude self-loops
    edge_pairs = list(zip(rows, cols))    
    
    tcr_sim_edge_index = torch.tensor(edge_pairs, dtype=torch.long).T  # shape [2, num_edges]

    # Similarity = 1 - distance
    edge_weights = 1 - chephy_matrix[rows, cols]
    edge_weights = torch.tensor(edge_weights, dtype=torch.float)
    edge_scores = torch.tensor(data['vdjdb.score'].values, dtype=torch.float)
        
    data_hetero = HeteroData()

    data_hetero['tcr'].x = tcr_features
    data_hetero['epitope'].x = epitope_features
    data_hetero['tcr', 'binds', 'epitope'].edge_index = edge_index
    data_hetero['tcr', 'binds', 'epitope'].edge_attr = edge_scores
    data_hetero['tcr', 'similar', 'tcr'].edge_index = tcr_sim_edge_index
    data_hetero['tcr', 'similar', 'tcr'].edge_attr = edge_weights


    num_tcr = len(sequences_list)
    num_epitopes = len(epitopes)
    # [num_tcr, num_epitopes], multi-label format
    tcr_labels = torch.zeros((num_tcr, num_epitopes), dtype=torch.float)

    for row in data.itertuples():
        tcr_idx = row.tcr_index
        epi_idx = row.epitope_index
        if tcr_idx != -1 and epi_idx != -1:
            tcr_labels[tcr_idx, epi_idx] = 1.0

    data_hetero['tcr'].y = tcr_labels  # shape: [num_tcr, num_epitopes]


    # create masks for training, validation and testing
    valid_indices = (tcr_labels.sum(dim=1) > 0).nonzero(as_tuple=True)[0]
    train_idx, temp_idx = train_test_split(valid_indices, test_size=0.3, random_state=42)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

    num_tcr = data_hetero['tcr'].num_nodes
    train_mask = torch.zeros(num_tcr, dtype=torch.bool)
    val_mask = torch.zeros(num_tcr, dtype=torch.bool)
    test_mask = torch.zeros(num_tcr, dtype=torch.bool)

    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True
    data_hetero['tcr'].train_mask = train_mask
    data_hetero['tcr'].val_mask = val_mask
    data_hetero['tcr'].test_mask = test_mask

    return data_hetero


In [ ]:
input_file = "../files/vdjdb_cdr3.csv"
max_dist=1
label_header = "antigen.epitope"
cdr3_header = "cdr3"

data = run.load_dataframe(input_file)

# data = read_csv_files_from_folder()

data = run.prepare_data(data, cdr3_header, label_header)    
data = cpm.truncate_sequences(data, cdr3_header, 4, 4)

sequences = set(data["cdr3_truncated"])
epitopes = list(set(data["antigen.epitope"]))

chephy_matrix, hamming_matrix, sequences_list = cpm.find_che_phy_dist(sequences)

data['tcr_index'] = data['cdr3_truncated'].apply(lambda x: sequences_list.index(x) if x in sequences_list else -1)
data['epitope_index'] = data['antigen.epitope'].apply(lambda x: epitopes.index(x) if x in epitopes else -1)

sequence_indices = np.arange(len(sequences_list))


In [ ]:
data_hetero = build_hetero_graph(data, chephy_matrix, hamming_matrix, sequences_list, epitopes)

In [ ]:
# add reverse edged from epitope to tcr
data_hetero = ToUndirected()(data_hetero)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_curve, auc
from torch_geometric.nn import HeteroConv, SAGEConv, Linear

# === Step 6: Define the GNN Model ===
class TCREpitopeGNN(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.convs = HeteroConv({
            ('tcr', 'similar', 'tcr'): SAGEConv((-1, -1), hidden_dim),
            ('tcr', 'binds', 'epitope'): SAGEConv((-1, -1), hidden_dim),
            ('epitope', 'rev_binds', 'tcr'): SAGEConv((-1, -1), hidden_dim),
        }, aggr='sum')

        self.linear = nn.ModuleDict({
            'tcr': Linear(hidden_dim, hidden_dim),
            'epitope': Linear(hidden_dim, hidden_dim),
        })

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.convs(x_dict, edge_index_dict)
        return {k: torch.relu(self.linear[k](x)) for k, x in x_dict.items()}

    def decode(self, z_tcr, z_epi, edge_index):
        src, dst = edge_index
        return (z_tcr[src] * z_epi[dst]).sum(dim=1)


# === Step 7: Training Loop ===
def sample_negative_edges(num_tcr, num_epi, positive_edges, num_samples):
    pos_set = set((int(i), int(j)) for i, j in zip(*positive_edges))
    neg_edges = set()
    while len(neg_edges) < num_samples:
        i = np.random.randint(0, num_tcr)
        j = np.random.randint(0, num_epi)
        if (i, j) not in pos_set:
            neg_edges.add((i, j))
    return torch.tensor(list(neg_edges), dtype=torch.long).T

def train_link_prediction(data_hetero, model, optimizer, loss_fn, device, epochs=20):
    data_hetero = data_hetero.to(device)
    model.to(device)
    model.train()

    num_tcr = data_hetero['tcr'].num_nodes
    num_epi = data_hetero['epitope'].num_nodes
    pos_edge_index = data_hetero['tcr', 'binds', 'epitope'].edge_index

    best_model_state = None
    best_val_loss = float('inf')

    for epoch in range(epochs):
        optimizer.zero_grad()
        z_dict = model(data_hetero.x_dict, data_hetero.edge_index_dict)

        neg_edge_index = sample_negative_edges(num_tcr, num_epi, pos_edge_index, pos_edge_index.size(1))

        edge_index = torch.cat([
            pos_edge_index.to(device),
            neg_edge_index.to(device)
        ], dim=1)

        edge_labels = torch.cat([
            torch.ones(pos_edge_index.size(1), device=device),
            torch.zeros(neg_edge_index.size(1), device=device)
        ]).to(device)

        scores = model.decode(z_dict['tcr'], z_dict['epitope'], edge_index)
        loss = loss_fn(scores, edge_labels)
        loss.backward()
        optimizer.step()
        if loss.item() < best_val_loss:
            best_val_loss = loss.item()
            best_model_state = model.state_dict()
        print(f"[Epoch {epoch+1}] Loss: {loss.item():.4f}")

    return best_model_state

# === Step 8: Evaluation ===
def partial_auc(y_true, y_scores, max_fpr=0.1):
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    mask = fpr <= max_fpr
    if mask.sum() < 2:
        return 0.0
    return auc(fpr[mask], tpr[mask]) / max_fpr

def macro_auc_0_1(tcr_embeddings, epitope_embeddings, test_edges, test_labels, test_epitope_ids):
    scores = (tcr_embeddings[test_edges[0]] * epitope_embeddings[test_edges[1]]).sum(dim=1).cpu().numpy()
    y_true = test_labels.cpu().numpy()
    epi_ids = test_epitope_ids.cpu().numpy()

    per_epi_auc = []
    for epi in np.unique(epi_ids):
        mask = epi_ids == epi
        if mask.sum() >= 5:
            
            auc_score = partial_auc(y_true[mask], scores[mask])
            per_epi_auc.append(auc_score)

    return np.mean(per_epi_auc) if per_epi_auc else 0.0


In [ ]:
# Set up model, optimizer, loss
model = TCREpitopeGNN(hidden_dim=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = torch.nn.BCEWithLogitsLoss()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
best_model_state = train_link_prediction(data_hetero, model, optimizer, loss_fn, device, epochs=20)

In [ ]:
def prepare_balanced_test_set(data, test_tcrs, num_tcr, num_epi, positive_edges, max_neg_ratio=3, min_pos=5, min_neg=5):
    """
    Create a balanced test set with valid epitopes only.
    """
    df_test = data[data['tcr_index'].isin(test_tcrs)]
    
    all_pos = []
    all_neg = []
    all_labels = []
    all_epitope_ids = []

    for epitope_id in data['epitope_index'].unique():
        df_epi = df_test[df_test['epitope_index'] == epitope_id]
        
        # Positive edges (from the data)
        df_pos = df_epi[df_epi['vdjdb.score'] > 0]
        pos = df_pos[['tcr_index', 'epitope_index']].drop_duplicates().values

        # Negative candidates (not in positive set)
        pos_set = set(map(tuple, pos))
        all_test_tcrs = set(df_epi['tcr_index'])
        neg_candidates = [(tcr, epitope_id) for tcr in all_test_tcrs if (tcr, epitope_id) not in pos_set]
        
        # Skip if too few positives or negatives
        if len(pos) < min_pos or len(neg_candidates) < min_neg:
            continue

        # Rebalance: limit negatives to max_neg_ratio × #positives
        np.random.shuffle(neg_candidates)
        neg = neg_candidates[:max_neg_ratio * len(pos)]

        # Add to pool
        all_pos.extend(pos)
        all_neg.extend(neg)
        all_labels.extend([1] * len(pos) + [0] * len(neg))
        all_epitope_ids.extend([epitope_id] * (len(pos) + len(neg)))

    # Final tensors
    if not all_pos:
        raise ValueError("No valid epitopes with enough pos/neg samples in test set.")

    test_pairs = torch.tensor(all_pos + all_neg, dtype=torch.long).T
    test_labels = torch.tensor(all_labels, dtype=torch.float)
    test_epitope_ids = torch.tensor(all_epitope_ids, dtype=torch.long)

    return test_pairs, test_labels, test_epitope_ids


In [ ]:
test_pairs, test_labels, test_epitope_ids = prepare_balanced_test_set(
    data,
    test_tcrs=data_hetero['tcr'].test_mask.nonzero(as_tuple=True)[0].tolist(),
    num_tcr=data_hetero['tcr'].num_nodes,
    num_epi=data_hetero['epitope'].num_nodes,
    positive_edges=data_hetero['tcr', 'binds', 'epitope'].edge_index,
    max_neg_ratio=3,
    min_pos=5,
    min_neg=5
)

In [ ]:
# (E) Evaluate the model
model.load_state_dict(best_model_state)
model.eval()
with torch.no_grad():
    z_dict = model(data_hetero.x_dict, data_hetero.edge_index_dict)
    auc_macro_01 = macro_auc_0_1(z_dict['tcr'], z_dict['epitope'], test_pairs, test_labels, test_epitope_ids)

print("Macro AUC@0.1:", auc_macro_01)